# unit cell packing prototype

How to choose shifts to apply to each image in order to optimally pack the unit cell?

Exercises:

1. Compute center of mass of the ASU in fractional coordinates
2. Check that repacking is compatible with neighbor search

### Ex. 1

Compute center of mass of the ASU in fractional coordinates

In [ ]:
import gemmi
st = gemmi.read_structure('test_data/lys_1_refmac.pdb')
st.setup_entities() # supposed to be good practice
st[0].remove_ligands_and_waters()


In [3]:
asu_com = st.cell.fractionalize(st[0].calculate_center_of_mass())

print(asu_com)
for im in st.cell.images:
    im_com = im.apply(asu_com)
    print(im_com)

<gemmi.Fractional(-0.00776163, 0.259636, 0.506122)>
<gemmi.Fractional(0.240364, 0.492238, 1.25612)>
<gemmi.Fractional(0.00776163, -0.259636, 1.00612)>
<gemmi.Fractional(0.759636, 0.507762, 0.756122)>
<gemmi.Fractional(0.492238, 0.240364, -0.256122)>
<gemmi.Fractional(-0.259636, 0.00776163, -0.00612246)>
<gemmi.Fractional(0.507762, 0.759636, 0.243878)>
<gemmi.Fractional(0.259636, -0.00776163, -0.506122)>


In [4]:
# now repack the unit cell
asu_shift = asu_com.wrap_to_unit() - asu_com
for im in st.cell.images:
    im_com = im.apply(asu_com)
    im_shift = im_com.wrap_to_unit() - im_com
    im.vec.x += im_shift.x - asu_shift.x
    im.vec.y += im_shift.y - asu_shift.y
    im.vec.z += im_shift.z - asu_shift.z

In [5]:
print(asu_com)
for im in st.cell.images:
    im_com = im.apply(asu_com)
    print(im_com)

<gemmi.Fractional(-0.00776163, 0.259636, 0.506122)>
<gemmi.Fractional(-0.759636, 0.492238, 0.256122)>
<gemmi.Fractional(-0.992238, 0.740364, 0.00612246)>
<gemmi.Fractional(-0.240364, 0.507762, 0.756122)>
<gemmi.Fractional(-0.507762, 0.240364, 0.743878)>
<gemmi.Fractional(-0.259636, 0.00776163, 0.993878)>
<gemmi.Fractional(-0.492238, 0.759636, 0.243878)>
<gemmi.Fractional(-0.740364, 0.992238, 0.493878)>


In [6]:
for im in st.cell.images:
    print(im.vec)

<gemmi.Vec3(-0.5, 0.5, -0.25)>
<gemmi.Vec3(-1, 1, -0.5)>
<gemmi.Vec3(-0.5, 0.5, 0.25)>
<gemmi.Vec3(-0.5, 0.5, 1.25)>
<gemmi.Vec3(0, 0, 1.5)>
<gemmi.Vec3(-0.5, 0.5, 0.75)>
<gemmi.Vec3(-1, 1, 1)>


### Ex. 2

Internal vs. external contacts in repacked cell

In [7]:
import pandas as pd

cs = gemmi.ContactSearch(4.0)
cs.ignore = gemmi.ContactSearch.Ignore.SameAsu # WARNING! will break for P1, and other low symmetry space groups
cs.twice=True # get both copies?
ns = gemmi.NeighborSearch(st[0], st.cell, 5).populate(include_h=False)
results = cs.find_contacts(ns)

def results2dict(contacts):
    d = {
        'cra1':[str(res.partner1) for res in contacts],
        'cra2':[str(res.partner2) for res in contacts],
        'sym_idx1':[0 for res in contacts],
        'sym_idx2':[res.image_idx for res in contacts],
        'pbc_shift1':[(0,0,0) for res in contacts],
        'pbc_shift2':[st.cell.find_nearest_pbc_image(
            res.partner1.atom.pos, 
            res.partner2.atom.pos, 
            res.image_idx).pbc_shift for res in contacts],
    }
    return d

df = pd.DataFrame.from_dict(results2dict(results))
df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2
0,A/ALA 10/CB,A/ARG 14/CD,0,5,"(0, 0, 0)","(0, 0, -1)"
1,A/LYS 13/CE,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, -1)"
2,A/LYS 13/CE,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, -1)"
3,A/LYS 13/NZ,A/LEU 129/O,0,5,"(0, 0, 0)","(0, 0, -1)"
4,A/LYS 13/NZ,A/LEU 129/C,0,5,"(0, 0, 0)","(0, 0, -1)"
...,...,...,...,...,...,...
197,A/LEU 129/C,A/LYS 13/CE,0,5,"(0, 0, 0)","(0, 0, -1)"
198,A/LEU 129/C,A/LYS 13/NZ,0,5,"(0, 0, 0)","(0, 0, -1)"
199,A/LEU 129/O,A/LYS 13/CE,0,5,"(0, 0, 0)","(0, 0, -1)"
200,A/LEU 129/O,A/LYS 13/NZ,0,5,"(0, 0, 0)","(0, 0, -1)"


In [8]:
df['interface'] = df.groupby(['sym_idx1','pbc_shift1','sym_idx2','pbc_shift2']).ngroup()
df2 = df.drop_duplicates(subset=['interface'], keep='first').set_index('interface')[['sym_idx2','pbc_shift2']].sort_index()
df2

,sym_idx2,pbc_shift2
interface,,
0,1,"(1, 0, 0)"
1,1,"(1, 0, 1)"
2,3,"(0, 0, -1)"
3,3,"(0, 0, 0)"
4,5,"(0, 0, -1)"
5,7,"(1, -1, 0)"
6,7,"(1, -1, 1)"


In [9]:
neighbor_ops = []

for index, row in df2.iterrows():
    #print(index, row['sym_idx2'], row['pbc_shift2'])
    if row['sym_idx2'] == 0:
        neighbor_op = gemmi.Transform()
    else:
        image_transform = st.cell.images[row['sym_idx2'] - 1]
        pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*row['pbc_shift2']))
        neighbor_op = pbc_transform.combine(image_transform)
    neighbor_ops.append(neighbor_op)
    
neighbor_ops

In [10]:
def is_identity(t,wrap=True,tol=1e-9):
    """return True if the transform is identity modulo PBC"""
    tf = False
    if t.mat.approx(gemmi.Mat33(),tol):
        fvec = gemmi.Fractional(*t.vec.tolist())
        if wrap:
            fvec = fvec.wrap_to_zero()
        tf = fvec.approx(gemmi.Vec3(0,0,0),tol)
    return tf

def find_inverse(t, images, wrap=True):
    """Find the inverse of a transform (t) from within a list of transforms (images). Returns the first matching index"""
    for j, im in enumerate(images):
        if is_identity(t.combine(im), wrap=wrap):
            return j

neighbor_pairs = [(j, find_inverse(t, neighbor_ops, wrap=False)) for j, t in enumerate(neighbor_ops)]
neighbor_pairs

[(0, 3), (1, 2), (2, 1), (3, 0), (4, 4), (5, 5), (6, 6)]

## Ex. 3 Putting together into a function

In [6]:
import gemmi

coordinate_file = 'test_data/lys_1_refmac.pdb'

st = gemmi.read_structure(coordinate_file)
st.setup_entities()  # supposed to be good practice
st[0].remove_ligands_and_waters()
asu_com = st.cell.fractionalize(st[0].calculate_center_of_mass())
asu_shift = asu_com.wrap_to_unit() - asu_com
for im in st.cell.images:
    im_com = im.apply(asu_com)
    im_shift = im_com.wrap_to_unit() - im_com
    im.vec.x += im_shift.x - asu_shift.x
    im.vec.y += im_shift.y - asu_shift.y
    im.vec.z += im_shift.z - asu_shift.z

for im in st.cell.images:
    print(im.mat, im.vec)

<gemmi.Mat33 [0, -1, 0]
             [1, 0, 0]
             [0, 0, 1]> <gemmi.Vec3(-0.5, 0.5, -0.25)>
<gemmi.Mat33 [-1, 0, 0]
             [0, -1, 0]
             [0, 0, 1]> <gemmi.Vec3(-1, 1, -0.5)>
<gemmi.Mat33 [0, 1, 0]
             [-1, 0, 0]
             [0, 0, 1]> <gemmi.Vec3(-0.5, 0.5, 0.25)>
<gemmi.Mat33 [1, 0, 0]
             [0, -1, 0]
             [0, 0, -1]> <gemmi.Vec3(-0.5, 0.5, 1.25)>
<gemmi.Mat33 [0, -1, 0]
             [-1, 0, 0]
             [0, 0, -1]> <gemmi.Vec3(0, 0, 1.5)>
<gemmi.Mat33 [-1, 0, 0]
             [0, 1, 0]
             [0, 0, -1]> <gemmi.Vec3(-0.5, 0.5, 0.75)>
<gemmi.Mat33 [0, 1, 0]
             [1, 0, 0]
             [0, 0, -1]> <gemmi.Vec3(-1, 1, 1)>


In [30]:
m2 = m.clone()
print(m.calculate_mass())
print(m2.calculate_mass())
m2.remove_hydrogens()
print(m.calculate_mass())
print(m2.calculate_mass())

14275.839520028156
14275.839520028156
14275.839520028156
13330.391800178355
